# spain-demographic-dynamics-map — Build GeoJSON Layers
## Step 1 of the web mapping pipeline (Matt Forrest / Modern GIS Accelerator methodology)

This notebook is **only** about producing the 4 GeoJSON layers that feed the rest of the
pipeline (tippecanoe / MapLibre / GitHub Pages are handled in separate steps, outside this
notebook):

| Layer      | Geometry source                          | Aggregation                     |
|------------|-------------------------------------------|----------------------------------|
| municipio  | original municipal polygons               | none (atomic unit)               |
| comarca    | dissolved by Comarca_Code                  | sum population per year          |
| provincia  | dissolved by Prov_Code                     | sum population per year          |
| ccaa       | dissolved by CCAA_Code                     | sum population per year          |

**Design decisions carried over from our discussion:**
- Only **raw population per year** goes into the properties (no precomputed % variation —
  that's computed client-side in JS so the year-range slider is fully free).
- Aggregation is done on raw population **sums**, never by averaging percentages — you can't
  average a % change across municipalities of very different size.
- **Ceuta and Melilla have no Comarca** (single-municipality provinces/CCAA). In the comarca
  layer, they are kept as their own municipal geometry rather than dissolved away.
- Each level gets progressively coarser geometry simplification (municipio finest, ccaa
  coarsest), since a CCAA-level view doesn't need coastline detail.
- The municipio layer carries two independent capital flags — `is_provincial_capital`
  (52 municipalities) and `is_ccaa_capital` (20 municipalities, since Canarias has two
  co-capitals) — kept separate rather than merged, so the frontend can style three distinct
  tiers (CCAA capital > provincial capital > regular municipality).

In [37]:
"""
Notebook: 00_build_population_geojson_layers.ipynb
Author: Juan Zotes
Project: spain-demographic-dynamics-map (portfolio / Modern GIS Accelerator course project,
         separate from RURIMESCAPE paper deliverables)

Purpose:
    Build the 4 GeoJSON layers (municipio, comarca, provincia, ccaa) used by the interactive
    population-variation web map. Each feature carries raw population per year only; %
    variation for an arbitrary year range is computed client-side in MapLibre/JS.

Input:
    - data/raw/mun_geographic_administrative_hierarchy.gpkg
    - data/raw/01_padron_clean_1996_2025.csv   (long format: Mun_Code, Mun, Cat, Year, Pop)

Output (data/processed/):
    - municipio.geojson   (includes is_provincial_capital, is_ccaa_capital flags)
    - comarca.geojson
    - provincia.geojson
    - ccaa.geojson

Notes:
    - This notebook does NOT run tippecanoe and does NOT touch the frontend (index.html).
      Those are separate steps.
    - Environment: 'gis' conda env (course environment), kept separate from 'rural-migration'
      since this is an independent portfolio project.
"""

"\nNotebook: 00_build_population_geojson_layers.ipynb\nAuthor: Juan Zotes\nProject: spain-demographic-dynamics-map (portfolio / Modern GIS Accelerator course project,\n         separate from RURIMESCAPE paper deliverables)\n\nPurpose:\n    Build the 4 GeoJSON layers (municipio, comarca, provincia, ccaa) used by the interactive\n    population-variation web map. Each feature carries raw population per year only; %\n    variation for an arbitrary year range is computed client-side in MapLibre/JS.\n\nInput:\n    - data/raw/mun_geographic_administrative_hierarchy.gpkg\n    - data/raw/01_padron_clean_1996_2025.csv   (long format: Mun_Code, Mun, Cat, Year, Pop)\n\nOutput (data/processed/):\n    - municipio.geojson   (includes is_provincial_capital, is_ccaa_capital flags)\n    - comarca.geojson\n    - provincia.geojson\n    - ccaa.geojson\n\nNotes:\n    - This notebook does NOT run tippecanoe and does NOT touch the frontend (index.html).\n      Those are separate steps.\n    - Environment: 'g

## 1. Imports and paths

In [38]:
from pathlib import Path
import json

import geopandas as gpd
import pandas as pd

# ---- Project folder ----
PROJECT_DIR = Path(
    r"C:\Users\juanz\OneDrive\Desktop\UCM\RURIM ESCAPE\GeoSpatial\00_Visualizaciones\spain-demographic-dynamics-map"
)

RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

DERIVED_DIR = PROJECT_DIR / "data" / "derived"
DERIVED_DIR.mkdir(parents=True, exist_ok=True)  # convert.sh also writes here; the
# notebook only needs it for search_index.json, a runtime frontend asset (not a
# build intermediate), so it belongs alongside the .pmtiles, not in data/processed/

# ---- Input files ----
FP_HIERARCHY_GPKG = RAW_DIR / "mun_geographic_administrative_hierarchy.gpkg"
FP_PADRON_CSV = RAW_DIR / "01_padron_clean_1996_2025.csv"

# ---- Output files ----
# Pipeline stages: raw (sources) -> processed (these 4 GeoJSON) -> derived
# (pmtiles + everything from convert.sh onward, produced outside this notebook)
FP_OUT_MUNICIPIO = PROCESSED_DIR / "municipio.geojson"
FP_OUT_COMARCA = PROCESSED_DIR / "comarca.geojson"
FP_OUT_PROVINCIA = PROCESSED_DIR / "provincia.geojson"
FP_OUT_CCAA = PROCESSED_DIR / "ccaa.geojson"

print(f"Raw dir       : {RAW_DIR}")
print(f"Processed dir : {PROCESSED_DIR}")
print(f"Hierarchy     : {FP_HIERARCHY_GPKG}")
print(f"Padron CSV    : {FP_PADRON_CSV}")

Raw dir       : C:\Users\juanz\OneDrive\Desktop\UCM\RURIM ESCAPE\GeoSpatial\00_Visualizaciones\spain-demographic-dynamics-map\data\raw
Processed dir : C:\Users\juanz\OneDrive\Desktop\UCM\RURIM ESCAPE\GeoSpatial\00_Visualizaciones\spain-demographic-dynamics-map\data\processed
Hierarchy     : C:\Users\juanz\OneDrive\Desktop\UCM\RURIM ESCAPE\GeoSpatial\00_Visualizaciones\spain-demographic-dynamics-map\data\raw\mun_geographic_administrative_hierarchy.gpkg
Padron CSV    : C:\Users\juanz\OneDrive\Desktop\UCM\RURIM ESCAPE\GeoSpatial\00_Visualizaciones\spain-demographic-dynamics-map\data\raw\01_padron_clean_1996_2025.csv


## 2. Load administrative hierarchy + geometry

In [39]:
gdf_hierarchy = gpd.read_file(FP_HIERARCHY_GPKG)
gdf_hierarchy["Mun_Code"] = gdf_hierarchy["Mun_Code"].astype(str).str.zfill(5)

print(f"Municipalities loaded : {len(gdf_hierarchy)}")
print(f"CRS                   : {gdf_hierarchy.crs}")
print(f"Columns               : {list(gdf_hierarchy.columns)}")

n_no_comarca = gdf_hierarchy["Comarca_Name"].isna().sum()
print(f"\nMunicipalities with no Comarca (expected: Ceuta, Melilla) : {n_no_comarca}")
gdf_hierarchy.loc[gdf_hierarchy["Comarca_Name"].isna(), ["Mun_Code", "Mun_Name", "Prov_Name"]]

Municipalities loaded : 8132
CRS                   : EPSG:4258
Columns               : ['Mun_Code', 'Mun_Name', 'Comarca_Code', 'Comarca_Name', 'Prov_Code', 'Prov_Name', 'CCAA_Code', 'CCAA_Name', 'nationalcode', 'geometry']

Municipalities with no Comarca (expected: Ceuta, Melilla) : 0


,Mun_Code,Mun_Name,Prov_Name


### 2.1 Fix Comarca_Name casing

`Comarca_Name` comes from the MAPA WFS source in ALL CAPS (e.g. `TIERRA DE CAMPOS`), which
visually overwhelms the municipality name in the popup label. We convert it to proper title
case here, once, before it propagates into the `municipio` layer (as the `comarca` property)
and the `comarca` layer (as `name`). Spanish connector words (de, la, el, y, ...) are kept
lowercase unless they're the first word.

In [40]:
SPANISH_LOWERCASE_WORDS = {"de", "del", "la", "las", "el", "los", "y", "en", "a"}


def spanish_title_case(text):
    """Title-case a Spanish place name, keeping connector words lowercase
    unless they're the first word (e.g. 'TIERRA DE CAMPOS' -> 'Tierra de Campos').
    """
    if pd.isna(text):
        return text
    words = str(text).strip().split()
    result = []
    for i, w in enumerate(words):
        if i > 0 and w.lower() in SPANISH_LOWERCASE_WORDS:
            result.append(w.lower())
        else:
            result.append(w.capitalize())
    return " ".join(result)


print("Before:", gdf_hierarchy["Comarca_Name"].dropna().unique()[:5].tolist())

gdf_hierarchy["Comarca_Name"] = gdf_hierarchy["Comarca_Name"].apply(spanish_title_case)

print("After: ", gdf_hierarchy["Comarca_Name"].dropna().unique()[:5].tolist())

Before: ['RIO NACIMIENTO', 'CAMPO DE DALIAS', 'ALTO ALMANZORA', 'ALTO ANDARAX', 'CAMPO DE TABERNAS']
After:  ['Rio Nacimiento', 'Campo de Dalias', 'Alto Almanzora', 'Alto Andarax', 'Campo de Tabernas']


## 3. Load population (long format) and pivot to wide

The padrón CSV is long format: `Mun_Code, Mun, Cat, Year, Pop`, with `Cat` in
`{Hombres, Mujeres, Total}`. We only need `Total`. `Mun_Code` **must** be read as string —
it has leading zeros (e.g. `01001`) that a numeric read would silently strip.

In [41]:
df_padron = pd.read_csv(
    FP_PADRON_CSV,
    dtype={"Mun_Code": str},
    sep=",",
    encoding="utf-8-sig",
)

df_padron["Mun_Code"] = df_padron["Mun_Code"].str.zfill(5)

print(f"Rows loaded        : {len(df_padron)}")
print(f"Categories present : {df_padron['Cat'].unique().tolist()}")
print(f"Mun_Code length check (should all be 5): {df_padron['Mun_Code'].str.len().unique()}")
df_padron.head()

Rows loaded        : 706017
Categories present : ['Hombres', 'Mujeres', 'Total']
Mun_Code length check (should all be 5): [5]


,Mun_Code,Mun,Cat,Year,Pop
0,01001,Alegría-Dulantzi,Hombres,1996,640.0
1,01001,Alegría-Dulantzi,Mujeres,1996,594.0
2,01001,Alegría-Dulantzi,Total,1996,1234.0
3,01001,Alegría-Dulantzi,Hombres,1998,656.0
4,01001,Alegría-Dulantzi,Mujeres,1998,603.0


In [42]:
# Keep Total only, pivot to wide (one column per year)
df_total = df_padron[df_padron["Cat"] == "Total"].copy()

df_pop_wide = (
    df_total
    .pivot_table(index="Mun_Code", columns="Year", values="Pop", aggfunc="sum")
    .sort_index(axis=1)
)

df_pop_wide.columns = [str(int(c)) for c in df_pop_wide.columns]
df_pop_wide = df_pop_wide.reset_index()

year_cols = [c for c in df_pop_wide.columns if c != "Mun_Code"]
year_cols = sorted(year_cols, key=int)

print(f"Municipalities with population data : {df_pop_wide['Mun_Code'].nunique()}")
print(f"Year columns ({len(year_cols)}): {year_cols}")
df_pop_wide.head()

Municipalities with population data : 8132
Year columns (29): ['1996', '1998', '1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025']


,Mun_Code,1996,1998,1999,2000,2001,2002,2003,2004,2005,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,01001,1234.0,1259.0,1329.0,1401.0,1486.0,1598.0,1707.0,1919.0,2048.0,...,2856.0,2913.0,2887.0,2876.0,2935.0,2925.0,2960.0,2975.0,2971.0,2955.0
1,01002,9758.0,9753.0,9753.0,9720.0,9695.0,9594.0,9555.0,9512.0,9592.0,...,10260.0,10291.0,10286.0,10350.0,10264.0,10307.0,10281.0,10313.0,10330.0,10364.0
2,01003,1345.0,1365.0,1373.0,1416.0,1447.0,1479.0,1489.0,1487.0,1503.0,...,1502.0,1496.0,1483.0,1467.0,1442.0,1456.0,1431.0,1409.0,1381.0,1354.0
3,01004,1293.0,1317.0,1338.0,1338.0,1331.0,1339.0,1381.0,1451.0,1499.0,...,1840.0,1829.0,1831.0,1821.0,1800.0,1809.0,1810.0,1832.0,1856.0,1868.0
4,01006,139.0,145.0,153.0,146.0,158.0,167.0,170.0,162.0,175.0,...,234.0,239.0,219.0,227.0,223.0,231.0,235.0,232.0,247.0,236.0


## 4. Helper functions

Reused across all 4 layers: geometry simplification (with a level-specific tolerance) and
property renaming to short keys (keeps the GeoJSON/PMTiles light — this file is consumed by
JS, not read by humans, so verbose names aren't worth the extra bytes across ~8,000+
features).

In [43]:
def simplify_geometry(gdf, tolerance_m, metric_crs=3857):
    """Simplify geometry with a meaningful tolerance in meters, then return to EPSG:4326
    (required for MapLibre / GeoJSON on the web). Fixes any invalid geometries produced
    by simplification via buffer(0).
    """
    g = gdf.to_crs(metric_crs)
    g["geometry"] = g.geometry.simplify(tolerance=tolerance_m, preserve_topology=True)
    g = g.to_crs(4326)

    invalid_mask = ~g.geometry.is_valid
    if invalid_mask.any():
        g.loc[invalid_mask, "geometry"] = g.loc[invalid_mask, "geometry"].buffer(0)
        print(f"  Fixed {invalid_mask.sum()} invalid geometries with buffer(0)")

    return g


def export_and_check(gdf, filepath, expected_count=None):
    """Export to GeoJSON and print sanity checks.

    Expects the GeoDataFrame to already carry an integer 'fid' column (the
    real INE administrative code for that level) -- tippecanoe can only
    promote a NUMERIC attribute to a tile feature id via
    --use-attribute-for-id, which is why 'fid' is assigned explicitly per
    layer (Mun_Code / Comarca_Code / Prov_Code / CCAA_Code as int) right
    before calling this function, rather than an arbitrary running counter.

    Converting e.g. "01001" -> 1001 strips a leading zero, but this can
    never collide with a code that doesn't start with 0 (e.g. "11001" ->
    11001): the stripped version always has one fewer digit, so the two
    numeric ranges never overlap. Verified here explicitly rather than
    assumed.
    """
    assert "fid" in gdf.columns, "gdf must have an integer 'fid' column before export"
    assert pd.api.types.is_integer_dtype(gdf["fid"]), "'fid' must be integer dtype"
    assert gdf["fid"].nunique() == len(gdf), (
        f"fid collision detected: {len(gdf)} rows but only {gdf['fid'].nunique()} unique fid values"
    )

    gdf.to_file(filepath, driver="GeoJSON")

    with open(filepath, encoding="utf-8") as f:
        gj = json.load(f)

    n_features = len(gj["features"])
    size_mb = filepath.stat().st_size / 1_000_000

    print(f"✓ Exported: {filepath.name}")
    print(f"  Features : {n_features}" + (f" (expected {expected_count})" if expected_count else ""))
    print(f"  Size     : {size_mb:.2f} MB")
    print(f"  CRS      : {gdf.crs} (must be EPSG:4326)")
    print(f"  fid range: {gdf['fid'].min()}–{gdf['fid'].max()}, all unique ✓")
    print(f"  Sample properties: {gj['features'][0]['properties']}")

    assert gdf.crs.to_epsg() == 4326, "CRS must be EPSG:4326 for the web mapping pipeline"
    if expected_count is not None:
        assert n_features == expected_count, f"Expected {expected_count} features, got {n_features}"
    print()

## 5. Layer 1 — Municipio

The atomic unit. No aggregation, just geometry + hierarchy names + population per year.

In [44]:
MUNICIPIO_TOLERANCE_M = 50

keep_hierarchy_cols = [
    "Mun_Code", "Mun_Name", "Comarca_Name", "Prov_Name", "CCAA_Name", "geometry",
]

gdf_municipio = gdf_hierarchy[keep_hierarchy_cols].merge(
    df_pop_wide, on="Mun_Code", how="left", validate="1:1"
)

n_missing = gdf_municipio[year_cols[0]].isna().sum()
print(f"Municipalities missing population data: {n_missing}")
if n_missing > 0:
    print(gdf_municipio.loc[gdf_municipio[year_cols[0]].isna(), ["Mun_Code", "Mun_Name", "Prov_Name"]])

print(f"\nSimplifying geometry (tolerance={MUNICIPIO_TOLERANCE_M}m)...")
gdf_municipio = simplify_geometry(gdf_municipio, MUNICIPIO_TOLERANCE_M)

gdf_municipio.head()

Municipalities missing population data: 38
     Mun_Code                     Mun_Name               Prov_Name
102     04904                    Balanegra                 Almería
147     11903     San Martín del Tesorillo                   Cádiz
223     14901            Fuente Carreteros                 Córdoba
224     14902                La Guijarrosa                 Córdoba
279     18065               Dehesas Viejas                 Granada
289     18077                       Fornes                 Granada
312     18106                        Játar                 Granada
396     18914                  Valderrubio                 Granada
397     18915     Domingo Pérez de Granada                 Granada
398     18916             Torrenueva Costa                 Granada
478     21902            La Zarza-Perrunal                  Huelva
575     23905            Arroyo del Ojanco                    Jaén
676     29902  Villanueva de la Concepción                  Málaga
677     29903      

,Mun_Code,Mun_Name,Comarca_Name,Prov_Name,CCAA_Name,geometry,1996,1998,1999,2000,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,04001,Abla,Rio Nacimiento,Almería,Andalucía,"POLYGON ((-2.78585 37.09174, -2.78368 37.09552...",1529.0,1516.0,1519.0,1516.0,...,1294.0,1267.0,1235.0,1249.0,1248.0,1258.0,1247.0,1255.0,1272.0,1268.0
1,04002,Abrucena,Rio Nacimiento,Almería,Andalucía,"POLYGON ((-2.88868 37.09169, -2.88786 37.09586...",1460.0,1462.0,1460.0,1455.0,...,1208.0,1170.0,1150.0,1202.0,1183.0,1208.0,1221.0,1275.0,1235.0,1241.0
2,04003,Adra,Campo de Dalias,Almería,Andalucía,"POLYGON ((-3.14019 36.78779, -3.13817 36.78927...",20898.0,21016.0,21286.0,21505.0,...,24713.0,24697.0,24859.0,25148.0,25412.0,25501.0,25300.0,25195.0,25515.0,25501.0
3,04004,Albanchez,Alto Almanzora,Almería,Andalucía,"POLYGON ((-2.20218 37.31223, -2.20083 37.31231...",626.0,605.0,605.0,586.0,...,796.0,754.0,753.0,741.0,687.0,726.0,735.0,682.0,682.0,688.0
4,04005,Alboloduy,Rio Nacimiento,Almería,Andalucía,"POLYGON ((-2.71288 37.07817, -2.70772 37.08682...",817.0,813.0,811.0,812.0,...,613.0,610.0,621.0,616.0,609.0,612.0,615.0,596.0,585.0,579.0


In [45]:
rename_municipio = {
    "Mun_Code": "code",
    "Mun_Name": "name",
    "Comarca_Name": "comarca",
    "Prov_Name": "prov",
    "CCAA_Name": "ccaa",
}
for y in year_cols:
    rename_municipio[y] = f"p{y}"

gdf_municipio_final = gdf_municipio.rename(columns=rename_municipio)

pop_cols_short = [f"p{y}" for y in year_cols]
gdf_municipio_final[pop_cols_short] = gdf_municipio_final[pop_cols_short].round(0)

gdf_municipio_final.head()

,code,name,comarca,prov,ccaa,geometry,p1996,p1998,p1999,p2000,...,p2016,p2017,p2018,p2019,p2020,p2021,p2022,p2023,p2024,p2025
0,04001,Abla,Rio Nacimiento,Almería,Andalucía,"POLYGON ((-2.78585 37.09174, -2.78368 37.09552...",1529.0,1516.0,1519.0,1516.0,...,1294.0,1267.0,1235.0,1249.0,1248.0,1258.0,1247.0,1255.0,1272.0,1268.0
1,04002,Abrucena,Rio Nacimiento,Almería,Andalucía,"POLYGON ((-2.88868 37.09169, -2.88786 37.09586...",1460.0,1462.0,1460.0,1455.0,...,1208.0,1170.0,1150.0,1202.0,1183.0,1208.0,1221.0,1275.0,1235.0,1241.0
2,04003,Adra,Campo de Dalias,Almería,Andalucía,"POLYGON ((-3.14019 36.78779, -3.13817 36.78927...",20898.0,21016.0,21286.0,21505.0,...,24713.0,24697.0,24859.0,25148.0,25412.0,25501.0,25300.0,25195.0,25515.0,25501.0
3,04004,Albanchez,Alto Almanzora,Almería,Andalucía,"POLYGON ((-2.20218 37.31223, -2.20083 37.31231...",626.0,605.0,605.0,586.0,...,796.0,754.0,753.0,741.0,687.0,726.0,735.0,682.0,682.0,688.0
4,04005,Alboloduy,Rio Nacimiento,Almería,Andalucía,"POLYGON ((-2.71288 37.07817, -2.70772 37.08682...",817.0,813.0,811.0,812.0,...,613.0,610.0,621.0,616.0,609.0,612.0,615.0,596.0,585.0,579.0


### 5.1 Mark provincial capitals

`prov` does not always match the capital municipality's name (Asturias→Oviedo,
Cantabria→Santander, Araba/Álava→Vitoria-Gasteiz, Gipuzkoa→Donostia/San Sebastián,
Bizkaia→Bilbao, Illes Balears→Palma, Navarra→Pamplona/Iruña, La Rioja→Logroño), so an
explicit lookup table is used instead of comparing `name == prov`. Validated against the
actual `prov`/`name` values in this dataset — all 52 provinces map to a capital, and all 52
capital names exist exactly as written among the municipality names.

In [46]:
CAPITAL_BY_PROVINCE = {
    "A Coruña": "A Coruña",
    "Alacant/Alicante": "Alacant/Alicante",
    "Albacete": "Albacete",
    "Almería": "Almería",
    "Araba/Álava": "Vitoria-Gasteiz",
    "Asturias": "Oviedo",
    "Badajoz": "Badajoz",
    "Barcelona": "Barcelona",
    "Bizkaia": "Bilbao",
    "Burgos": "Burgos",
    "Cantabria": "Santander",
    "Castelló/Castellón": "Castelló de la Plana/Castellón de la Plana",
    "Ceuta": "Ceuta",
    "Ciudad Real": "Ciudad Real",
    "Cuenca": "Cuenca",
    "Cáceres": "Cáceres",
    "Cádiz": "Cádiz",
    "Córdoba": "Córdoba",
    "Gipuzkoa": "Donostia/San Sebastián",
    "Girona": "Girona",
    "Granada": "Granada",
    "Guadalajara": "Guadalajara",
    "Huelva": "Huelva",
    "Huesca": "Huesca",
    "Illes Balears": "Palma",
    "Jaén": "Jaén",
    "La Rioja": "Logroño",
    "Las Palmas": "Las Palmas de Gran Canaria",
    "León": "León",
    "Lleida": "Lleida",
    "Lugo": "Lugo",
    "Madrid": "Madrid",
    "Melilla": "Melilla",
    "Murcia": "Murcia",
    "Málaga": "Málaga",
    "Navarra": "Pamplona/Iruña",
    "Ourense": "Ourense",
    "Palencia": "Palencia",
    "Pontevedra": "Pontevedra",
    "Salamanca": "Salamanca",
    "Santa Cruz de Tenerife": "Santa Cruz de Tenerife",
    "Segovia": "Segovia",
    "Sevilla": "Sevilla",
    "Soria": "Soria",
    "Tarragona": "Tarragona",
    "Teruel": "Teruel",
    "Toledo": "Toledo",
    "Valladolid": "Valladolid",
    "València/Valencia": "València",
    "Zamora": "Zamora",
    "Zaragoza": "Zaragoza",
    "Ávila": "Ávila",
}

gdf_municipio_final["is_provincial_capital"] = gdf_municipio_final.apply(
    lambda row: row["name"] == CAPITAL_BY_PROVINCE.get(row["prov"]), axis=1
)

n_prov_capitals = gdf_municipio_final["is_provincial_capital"].sum()
print(f"Provincial capitals marked: {n_prov_capitals} (expected: 52)")
assert n_prov_capitals == 52, f"Expected 52 provincial capitals, found {n_prov_capitals}"

Provincial capitals marked: 52 (expected: 52)


### 5.2 Mark CCAA capitals (independent flag, not merged with provincial capital)

Kept as its own boolean rather than folded into `is_provincial_capital` — 15 of the 19 CCAA
capitals coincide with a provincial capital already marked above, but marking them explicitly
here means the frontend can style CCAA capitals as their own visual tier even where they
overlap with a provincial capital. **Canarias is a special case: it has two co-capitals**
(Santa Cruz de Tenerife and Las Palmas de Gran Canaria), both marked `True`.

In [47]:
CAPITAL_BY_CCAA = {
    "Andalucía": ["Sevilla"],
    "Aragón": ["Zaragoza"],
    "Canarias": ["Santa Cruz de Tenerife", "Las Palmas de Gran Canaria"],
    "Cantabria": ["Santander"],
    "Castilla y León": ["Valladolid"],
    "Castilla-La Mancha": ["Toledo"],
    "Cataluña/Catalunya": ["Barcelona"],
    "Ciudad Autónoma de Ceuta": ["Ceuta"],
    "Ciudad Autónoma de Melilla": ["Melilla"],
    "Comunidad Foral de Navarra": ["Pamplona/Iruña"],
    "Comunidad de Madrid": ["Madrid"],
    "Comunitat Valenciana": ["València"],
    "Extremadura": ["Mérida"],
    "Galicia": ["Santiago de Compostela"],
    "Illes Balears": ["Palma"],
    "La Rioja": ["Logroño"],
    "País Vasco/Euskadi": ["Vitoria-Gasteiz"],
    "Principado de Asturias": ["Oviedo"],
    "Región de Murcia": ["Murcia"],
}


def is_ccaa_capital(row):
    return row["name"] in CAPITAL_BY_CCAA.get(row["ccaa"], [])


gdf_municipio_final["is_ccaa_capital"] = gdf_municipio_final.apply(is_ccaa_capital, axis=1)

n_ccaa_capitals = gdf_municipio_final["is_ccaa_capital"].sum()
print(f"CCAA capitals marked: {n_ccaa_capitals} (expected: 20 — 19 CCAA, Canarias has 2 co-capitals)")
assert n_ccaa_capitals == 20, f"Expected 20 CCAA capital markers, found {n_ccaa_capitals}"

n_both = (gdf_municipio_final["is_provincial_capital"] & gdf_municipio_final["is_ccaa_capital"]).sum()
print(f"Municipalities that are both provincial AND CCAA capital: {n_both} (expected: 15)")

CCAA capitals marked: 20 (expected: 20 — 19 CCAA, Canarias has 2 co-capitals)
Municipalities that are both provincial AND CCAA capital: 18 (expected: 15)


### 5.3 Finalize column order and export

In [48]:
final_cols_municipio = (
    ["code", "name", "comarca", "prov", "ccaa"]
    + pop_cols_short
    + ["is_provincial_capital", "is_ccaa_capital", "geometry"]
)
gdf_municipio_final = gdf_municipio_final[final_cols_municipio]

# fid = Mun_Code as integer (real INE code, e.g. "01001" -> 1001). Verified
# unique across all 8,132 municipalities -- see conversation notes.
gdf_municipio_final["fid"] = gdf_municipio_final["code"].astype(int)

export_and_check(gdf_municipio_final, FP_OUT_MUNICIPIO, expected_count=8132)

✓ Exported: municipio.geojson
  Features : 8132 (expected 8132)
  Size     : 29.46 MB
  CRS      : EPSG:4326 (must be EPSG:4326)
  fid range: 1001–52001, all unique ✓
  Sample properties: {'code': '04001', 'name': 'Abla', 'comarca': 'Rio Nacimiento', 'prov': 'Almería', 'ccaa': 'Andalucía', 'p1996': 1529.0, 'p1998': 1516.0, 'p1999': 1519.0, 'p2000': 1516.0, 'p2001': 1517.0, 'p2002': 1529.0, 'p2003': 1480.0, 'p2004': 1482.0, 'p2005': 1512.0, 'p2006': 1505.0, 'p2007': 1514.0, 'p2008': 1503.0, 'p2009': 1504.0, 'p2010': 1463.0, 'p2011': 1480.0, 'p2012': 1465.0, 'p2013': 1422.0, 'p2014': 1426.0, 'p2015': 1342.0, 'p2016': 1294.0, 'p2017': 1267.0, 'p2018': 1235.0, 'p2019': 1249.0, 'p2020': 1248.0, 'p2021': 1258.0, 'p2022': 1247.0, 'p2023': 1255.0, 'p2024': 1272.0, 'p2025': 1268.0, 'is_provincial_capital': False, 'is_ccaa_capital': False, 'fid': 4001}



## 6. Layer 2 — Comarca

Dissolved by `Comarca_Code` (the real code from the source hierarchy), population summed
per year. `Comarca_Code` has zero nulls -- Ceuta and Melilla already carry their own
singleton codes (51, 52), so they fall out of a plain dissolve correctly with no
special-casing needed.

In [49]:
COMARCA_TOLERANCE_M = 100

# Comarca_Code has zero nulls in the source hierarchy (Ceuta=51, Melilla=52
# already have their own singleton codes), so a plain dissolve by Comarca_Code
# handles every municipality uniformly -- no Ceuta/Melilla special-case needed.
#
# 5 of the 338 comarcas span two provinces (verified against the source data),
# always with a heavy skew (1 municipality on one side, dozens on the other).
# We take the MAJORITY province (mode) as representative for the popup, rather
# than an arbitrary "first" match that could pick the minority side.
gdf_with_pop = gdf_hierarchy.merge(df_pop_wide, on="Mun_Code", how="left", validate="1:1")

def majority_prov(s):
    return s.value_counts().idxmax()

agg_dict_comarca = {y: "sum" for y in year_cols}
agg_dict_comarca["Comarca_Name"] = "first"
agg_dict_comarca["CCAA_Name"] = "first"  # a comarca stays within a single CCAA
agg_dict_comarca["Prov_Name"] = majority_prov  # see note above: 5/338 span 2 provinces

gdf_comarca = (
    gdf_with_pop
    .dissolve(by=["Comarca_Code"], aggfunc=agg_dict_comarca)
    .reset_index()
)

print(f"Comarca-level features (comarcas + Ceuta + Melilla as singletons): {len(gdf_comarca)}")

Comarca-level features (comarcas + Ceuta + Melilla as singletons): 338


In [50]:
keep_cols_comarca = ["Comarca_Code", "Comarca_Name", "Prov_Name", "CCAA_Name"] + year_cols + ["geometry"]
gdf_comarca = gpd.GeoDataFrame(gdf_comarca[keep_cols_comarca], geometry="geometry", crs=gdf_hierarchy.crs)

print(f"\nSimplifying geometry (tolerance={COMARCA_TOLERANCE_M}m)...")
gdf_comarca = simplify_geometry(gdf_comarca, COMARCA_TOLERANCE_M)

rename_comarca = {"Comarca_Name": "name", "Prov_Name": "prov", "CCAA_Name": "ccaa"}
for y in year_cols:
    rename_comarca[y] = f"p{y}"

gdf_comarca_final = gdf_comarca.rename(columns=rename_comarca)
gdf_comarca_final[pop_cols_short] = gdf_comarca_final[pop_cols_short].round(0)

# fid = Comarca_Code as integer (real code, e.g. 404 -> Almería's comarca 04).
# Comarca_Code came through the dissolve as the index; re-attach it explicitly.
gdf_comarca_final["fid"] = gdf_comarca["Comarca_Code"].astype(int).values
gdf_comarca_final["code"] = gdf_comarca_final["fid"]  # keep a visible code property too

gdf_comarca_final = gdf_comarca_final[["fid", "code", "name", "prov", "ccaa"] + pop_cols_short + ["geometry"]]

export_and_check(gdf_comarca_final, FP_OUT_COMARCA)


Simplifying geometry (tolerance=100m)...
✓ Exported: comarca.geojson
  Features : 338
  Size     : 5.12 MB
  CRS      : EPSG:4326 (must be EPSG:4326)
  fid range: 51–5007, all unique ✓
  Sample properties: {'fid': 51, 'code': 51, 'name': '', 'prov': 'Ceuta', 'ccaa': 'Ciudad Autónoma de Ceuta', 'p1996': 68796.0, 'p1998': 72117.0, 'p1999': 73704.0, 'p2000': 75241.0, 'p2001': 75694.0, 'p2002': 76152.0, 'p2003': 74931.0, 'p2004': 74654.0, 'p2005': 75276.0, 'p2006': 75861.0, 'p2007': 76603.0, 'p2008': 77389.0, 'p2009': 78674.0, 'p2010': 80579.0, 'p2011': 82376.0, 'p2012': 84018.0, 'p2013': 84180.0, 'p2014': 84963.0, 'p2015': 84263.0, 'p2016': 84519.0, 'p2017': 84959.0, 'p2018': 85144.0, 'p2019': 84777.0, 'p2020': 84202.0, 'p2021': 83517.0, 'p2022': 83117.0, 'p2023': 83039.0, 'p2024': 83229.0, 'p2025': 83595.0}



## 7. Layer 3 — Provincia

Dissolved by `Prov_Code`. Ceuta and Melilla are already single-municipality provinces, so
they fall out of the dissolve naturally — no special-casing needed here.

In [51]:
PROVINCIA_TOLERANCE_M = 200

agg_dict_prov = {y: "sum" for y in year_cols}
agg_dict_prov["Prov_Name"] = "first"
agg_dict_prov["CCAA_Name"] = "first"

gdf_provincia = (
    gdf_with_pop
    .dissolve(by=["Prov_Code"], aggfunc=agg_dict_prov)
    .reset_index()
)

print(f"Provinces after dissolve: {len(gdf_provincia)} (expected 52)")

print(f"\nSimplifying geometry (tolerance={PROVINCIA_TOLERANCE_M}m)...")
gdf_provincia = simplify_geometry(gdf_provincia, PROVINCIA_TOLERANCE_M)

rename_provincia = {"Prov_Name": "name", "CCAA_Name": "ccaa"}
for y in year_cols:
    rename_provincia[y] = f"p{y}"

gdf_provincia_final = gdf_provincia.rename(columns=rename_provincia)
gdf_provincia_final[pop_cols_short] = gdf_provincia_final[pop_cols_short].round(0)

# fid = Prov_Code as integer (real INE province code, e.g. "28" -> 28 = Madrid)
gdf_provincia_final["fid"] = gdf_provincia["Prov_Code"].astype(int).values
gdf_provincia_final["code"] = gdf_provincia_final["fid"]

gdf_provincia_final = gdf_provincia_final[["fid", "code", "name", "ccaa"] + pop_cols_short + ["geometry"]]

export_and_check(gdf_provincia_final, FP_OUT_PROVINCIA, expected_count=52)

Provinces after dissolve: 52 (expected 52)

Simplifying geometry (tolerance=200m)...
✓ Exported: provincia.geojson
  Features : 52 (expected 52)
  Size     : 1.93 MB
  CRS      : EPSG:4326 (must be EPSG:4326)
  fid range: 1–52, all unique ✓
  Sample properties: {'fid': 1, 'code': 1, 'name': 'Araba/Álava', 'ccaa': 'País Vasco/Euskadi', 'p1996': 281821.0, 'p1998': 284595.0, 'p1999': 285748.0, 'p2000': 286497.0, 'p2001': 288793.0, 'p2002': 291860.0, 'p2003': 294360.0, 'p2004': 295905.0, 'p2005': 299957.0, 'p2006': 301926.0, 'p2007': 305459.0, 'p2008': 309635.0, 'p2009': 313819.0, 'p2010': 317352.0, 'p2011': 319227.0, 'p2012': 322557.0, 'p2013': 321417.0, 'p2014': 321932.0, 'p2015': 323648.0, 'p2016': 324126.0, 'p2017': 326574.0, 'p2018': 328868.0, 'p2019': 331549.0, 'p2020': 333940.0, 'p2021': 333626.0, 'p2022': 334412.0, 'p2023': 336686.0, 'p2024': 339137.0, 'p2025': 342161.0}



## 8. Layer 4 — CCAA

Dissolved by `CCAA_Code`. Coarsest simplification tolerance — this is the zoomed-out
national view.

In [52]:
CCAA_TOLERANCE_M = 500

agg_dict_ccaa = {y: "sum" for y in year_cols}
agg_dict_ccaa["CCAA_Name"] = "first"

gdf_ccaa = (
    gdf_with_pop
    .dissolve(by=["CCAA_Code"], aggfunc=agg_dict_ccaa)
    .reset_index()
)

print(f"CCAA after dissolve: {len(gdf_ccaa)} (expected 19)")

print(f"\nSimplifying geometry (tolerance={CCAA_TOLERANCE_M}m)...")
gdf_ccaa = simplify_geometry(gdf_ccaa, CCAA_TOLERANCE_M)

rename_ccaa = {"CCAA_Name": "name"}
for y in year_cols:
    rename_ccaa[y] = f"p{y}"

gdf_ccaa_final = gdf_ccaa.rename(columns=rename_ccaa)
gdf_ccaa_final[pop_cols_short] = gdf_ccaa_final[pop_cols_short].round(0)

# fid = CCAA_Code as integer (real INE CCAA code, e.g. "13" -> 13 = Madrid)
gdf_ccaa_final["fid"] = gdf_ccaa["CCAA_Code"].astype(int).values
gdf_ccaa_final["code"] = gdf_ccaa_final["fid"]

gdf_ccaa_final = gdf_ccaa_final[["fid", "code", "name"] + pop_cols_short + ["geometry"]]

export_and_check(gdf_ccaa_final, FP_OUT_CCAA, expected_count=19)

CCAA after dissolve: 19 (expected 19)

Simplifying geometry (tolerance=500m)...
✓ Exported: ccaa.geojson
  Features : 19 (expected 19)
  Size     : 1.07 MB
  CRS      : EPSG:4326 (must be EPSG:4326)
  fid range: 1–19, all unique ✓
  Sample properties: {'fid': 1, 'code': 1, 'name': 'Andalucía', 'p1996': 7234797.0, 'p1998': 7236459.0, 'p1999': 7305117.0, 'p2000': 7340052.0, 'p2001': 7403968.0, 'p2002': 7478432.0, 'p2003': 7606848.0, 'p2004': 7687518.0, 'p2005': 7849799.0, 'p2006': 7975672.0, 'p2007': 8059461.0, 'p2008': 8202220.0, 'p2009': 8302923.0, 'p2010': 8370975.0, 'p2011': 8424102.0, 'p2012': 8449985.0, 'p2013': 8440300.0, 'p2014': 8402305.0, 'p2015': 8399043.0, 'p2016': 8388107.0, 'p2017': 8379820.0, 'p2018': 8384408.0, 'p2019': 8414240.0, 'p2020': 8464411.0, 'p2021': 8472407.0, 'p2022': 8500187.0, 'p2023': 8568513.0, 'p2024': 8619616.0, 'p2025': 8666412.0}



## 9. Build the search index

Lightweight index (name + type + centroid + breadcrumb) for the search box in the frontend.
Vector tiles only expose whatever is currently loaded in the viewport, so a search feature
needs its own small, separately-loaded index rather than depending on tile state -- this is
the standard pattern for search-then-fly-to on a PMTiles/vector-tile map.

Kept intentionally small: no population-by-year data here (that stays in the tiles only,
fetched fresh once the user flies to a result and its tile loads -- same popup/highlight
code path as a normal click, just triggered programmatically instead of by a mouse click).

In [53]:
def build_search_entries(gdf, layer_type, breadcrumb_cols):
    """One row per feature: type, code, name, breadcrumb fields, centroid (lon, lat).

    Centroid computed in a projected CRS (EPSG:3857) then reprojected back to
    EPSG:4326, same pattern as simplify_geometry -- more accurate than a naive
    centroid directly in lon/lat for large or irregular shapes.
    """
    g = gdf.to_crs(3857)
    centroids = g.geometry.centroid.to_crs(4326)

    entries = []
    for i, row in gdf.reset_index(drop=True).iterrows():
        entry = {
            "type": layer_type,
            "code": row["code"],
            "name": row["name"],
            "lon": round(centroids.iloc[i].x, 5),
            "lat": round(centroids.iloc[i].y, 5),
        }
        for col in breadcrumb_cols:
            entry[col] = row[col]
        entries.append(entry)
    return entries


search_entries = []
search_entries += build_search_entries(gdf_municipio_final, "municipio", ["comarca", "prov", "ccaa"])
search_entries += build_search_entries(gdf_comarca_final, "comarca", ["prov", "ccaa"])
search_entries += build_search_entries(gdf_provincia_final, "provincia", ["ccaa"])
search_entries += build_search_entries(gdf_ccaa_final, "ccaa", [])

FP_OUT_SEARCH_INDEX = DERIVED_DIR / "search_index.json"
with open(FP_OUT_SEARCH_INDEX, "w", encoding="utf-8") as f:
    json.dump(search_entries, f, ensure_ascii=False, separators=(",", ":"))

size_kb = FP_OUT_SEARCH_INDEX.stat().st_size / 1_000
print(f"✓ Exported: {FP_OUT_SEARCH_INDEX.name}")
print(f"  Entries : {len(search_entries)} (expected {8132 + 338 + 52 + 19})")
print(f"  Size    : {size_kb:.1f} KB")
print(f"  Sample  : {search_entries[0]}")

✓ Exported: search_index.json
  Entries : 8541 (expected 8541)
  Size    : 1326.1 KB
  Sample  : {'type': 'municipio', 'code': '04001', 'name': 'Abla', 'lon': -2.76673, 'lat': 37.16235, 'comarca': 'Rio Nacimiento', 'prov': 'Almería', 'ccaa': 'Andalucía'}


## 9. Summary

In [54]:
print("="*60)
print("ALL 4 LAYERS EXPORTED")
print("="*60)
for fp in [FP_OUT_MUNICIPIO, FP_OUT_COMARCA, FP_OUT_PROVINCIA, FP_OUT_CCAA]:
    size_mb = fp.stat().st_size / 1_000_000
    print(f"  {fp.name:20s} {size_mb:8.2f} MB")
print("\nNext step (separate, outside this notebook): tippecanoe (WSL) to combine")
print("all 4 layers into a single population_variation.pmtiles with -L per layer.")

ALL 4 LAYERS EXPORTED
  municipio.geojson       29.46 MB
  comarca.geojson          5.12 MB
  provincia.geojson        1.93 MB
  ccaa.geojson             1.07 MB

Next step (separate, outside this notebook): tippecanoe (WSL) to combine
all 4 layers into a single population_variation.pmtiles with -L per layer.
